<a href="https://colab.research.google.com/github/busycaesar/Embeddings_And_Cosine_Similarity/blob/Master/AgentCon%20-%20Toronto/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required dependencies

In [45]:
%pip install langchain_community unstructured azure-identity

Import environment variables

In [29]:
from google.colab import userdata
from google.colab import auth

AZURE_OPEN_API_ENDPOINT = userdata.get("AZURE_OPEN_API_ENDPOINT")
AZURE_OPEN_API_KEY = userdata.get("AZURE_OPEN_API_KEY")

VECTOR_SEARCH_ENDPOINT = userdata.get("VECTOR_SEARCH_ENDPOINT")
VECTOR_SEARCH_KEY = userdata.get("VECTOR_SEARCH_KEY")

auth.authenticate_user()

Fetch the data

In [30]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-900',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-300',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/dp-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-731',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-730',
]

loader = UnstructuredURLLoader(urls)

documents = loader.load()

print(len(documents))

7


Split the data into chunks

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

206


Store the data into vector database

In [42]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import AzureOpenAIEmbeddings, OpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="text-embedding-3-small",
)

vector_store = AzureSearch(
    azure_search_endpoint=VECTOR_SEARCH_ENDPOINT,
    azure_search_key=VECTOR_SEARCH_KEY,
    index_name='consine-similarity-demo',
    embedding_function=embeddings.embed_query
)

vector_store.add_documents(chunks)

['NTM2MjU5MWUtYWUwOC00OTJhLWEwMzgtMzFkZTY4MjE2NjA5',
 'M2UwOWI1OTgtODgwYy00MGQ3LThkZGQtYjFiOTg1ZDUyYjk5',
 'YzYzODg4OTQtNGVmZS00OGZjLWI5OTctM2QwYTZmYjllYTNm',
 'OTJhZTdhYWMtNDcyOC00YzQ3LWE4NWYtZTkzNzgzYTUwMzFi',
 'M2Q0NDMxMDItZTQyNC00YzFmLTg5YzMtZjUxNjU3MjZmYTI4',
 'YjQ2MmQxNzAtOTJjYy00ZjdiLTkzOGItMGQyNjRmODExMDhh',
 'NWY5NGVkZmUtNjNiNS00MGFkLTkwNDAtZGE1MGQ3YjhiNjE5',
 'YmZkYWU1YjEtNTYxOS00ZDllLTkwYmMtYzRiYzlhOWMyNmE2',
 'NzA1M2ViZDctNjQxZi00OGVkLWI4YTEtZWFiZjAxNGU2ZDgz',
 'MjhlYTIxMDItZGIxNC00MGUxLWFmN2QtMzJiZmE3NGJjNDMy',
 'Nzc5ZTk1NGItMDhhMi00ZjgwLTk2MGEtZWEzMzdjNjU4OTI4',
 'NjdiOTYwOTktZjAwNS00MDIyLThkYWQtNmY3YzE4MmU5MDE4',
 'OTVlMThmNjctYmRiZS00OWM3LWIyOGUtYWE5MDU0ZDIxZTlm',
 'NzY3ZjIwMTctYjYxZS00ZDEwLThlNjAtY2E5ZTU2YTY0ZjE5',
 'MDg1NDQ4MjAtZmVmMi00MDI0LTgwYWEtMjIzMWNjMmUyNTE1',
 'ZTcyNzA3NTktOWY0Yy00ZGNjLWIxZWMtMTZiOGJlODRlMzMy',
 'NWQ2ZGRlYTMtZTA1Ni00N2Y5LTk2NjYtMWE4ZjhiYmQyOTY2',
 'MTUwN2NmMGQtOWNlNy00NTA0LTg5ZDYtZGQ1MWQ3ZDE5MjAw',
 'YWRhOTdhMjUtYzcwMS00MmY1LWI5MDAtYWI3OTU2Zjgw

User's query

In [34]:
user_query = "What resource should I refer to study for ai-900?"

Fetch the relevant chunk of data

In [35]:
retrieved_docs = vector_store.similarity_search(query=user_query, k=3)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [36]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
        You are a helpful cloud instructor that provides cloud project ideas about Microsoft Azure Certifications based on the certification guide.

        Study Guide: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [44]:
from langchain_openai import AzureChatOpenAI
from IPython.display import clear_output

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="gpt-4o-mini",
    openai_api_version="2025-01-01-preview",
)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)

For studying for the AI-900: Microsoft Azure AI Fundamentals exam, you can utilize the following resources:

1. **Microsoft Documentation**: The official Microsoft Azure documentation is a comprehensive resource that covers the fundamentals of artificial intelligence and the tools available on Azure. Look for the "AI Fundamentals" section that might include tutorials, overviews of AI services, and best practices.

2. **Microsoft Learn**: This is an interactive platform offering modules and learning paths tailored for the AI-900 exam. Search for the AI-900 learning path, which includes step-by-step guides and hands-on labs.

3. **Practice Tests**: Websites that offer practice exams can help you gauge your knowledge and prepare for the format of the actual exam. Microsoft often provides sample questions in their learning resources.

4. **Books and E-books**: There are numerous books focused on Azure AI Fundamentals that can provide in-depth explanations and examples.

5. **Online Courses